# M1 — Data Collection & Preprocessing

**Project:** Customer Churn Prediction with Explainable AI  
**Dataset:** UCI Online Retail II  
**Owner:** Member 1

This notebook runs and documents the M1 pipeline implemented in `src/preprocessing.py`.

See also:
- `docs/dataset_source.md`
- `docs/preprocessing_decisions.md`

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import (
    load_raw,
    clean_transactions,
    build_customer_churn_dataset,
    validate_customer_dataset,
    run_pipeline,
    OBS_START,
    OBS_END,
    PRED_START,
    PRED_END,
    RAW_PATH,
)

print("ROOT:", ROOT)
print("RAW:", RAW_PATH)
print("Observation:", OBS_START.date(), "→", OBS_END.date())
print("Prediction:", PRED_START.date(), "→", PRED_END.date())

## 1. Load & inspect raw data

In [ ]:
raw = load_raw()
print("Shape:", raw.shape)
print("Columns:", list(raw.columns))
display(raw.dtypes.to_frame("dtype"))
display(raw.head())
print("Date range:", raw["InvoiceDate"].min(), "→", raw["InvoiceDate"].max())
display(raw.isna().sum().to_frame("missing"))

## 2. Clean transactions

Steps: drop missing Customer ID → drop cancellations → drop invalid qty/price → drop duplicates.

In [ ]:
cleaned, stats = clean_transactions(raw)
print(stats)
print("Cleaned shape:", cleaned.shape)
display(cleaned.head())

## 3. Customer-level churn labels

- Eligible: ≥1 purchase in observation period (2010-01-01 to 2010-12-31)
- `churn = 1` if zero purchases in prediction period (2011-01-01 to 2011-06-30)

In [ ]:
customers = build_customer_churn_dataset(cleaned)
display(customers.head())
print("Customers:", len(customers))
print("Churn rate:", customers["churn"].mean())
display(customers["churn"].value_counts().rename("count"))

## 4. Validate labels

In [ ]:
validation = validate_customer_dataset(customers, cleaned)
print(json.dumps(validation, indent=2))
assert validation["all_passed"], "Validation failed"

## 5. Write artifacts (full pipeline)

Prefer running this cell (or `python -m src.preprocessing`) to regenerate outputs under `data/processed/`.

In [ ]:
report = run_pipeline()
print(json.dumps(report, indent=2))
assert report["validation"]["all_passed"]